# ANA_EDU_FIN_CLI V1

ETL por ciclos financeiros fechados. Detalhes em `DOCUMENTACAO_ETL.md`.


## Fluxo

Cada bloco executa uma etapa do ETL e entrega dados para o bloco seguinte.


## 1. Conexão

Cria as sessões Spark local e remota e o cliente DB2.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()

    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="etl-vinculacao-mf-insights",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer":
                "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive":"true",
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
try:
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%%spark

import os

ambiente = ler_variavel_ambiente_spark("AMBIENTE").upper()
dominio = ler_variavel_ambiente_spark("DOMINIO").lower()
hoje = ler_variavel_ambiente_spark("HOJE")

if ambiente == "MODELAGEM":
    sandbox = ler_variavel_ambiente_spark("SANDBOX").lower()
    database = f"sbx_{sandbox}"
else:
    database = f"hive_{dominio}"

env_spark = dict(os.environ)

cliente_db2 = criar_cliente_db2_spark(env=env_spark)


## 2. Configuração

Define `PERIODOS`, a tabela de saída e a data de execução.


In [ ]:
%%spark

from datetime import date, timedelta
from time import perf_counter

nome_tabela = "ana_edu_fin_cli"
tabela_spark = f"{database}.{nome_tabela}"

PERIODOS = 2

data_execucao = date.fromisoformat(hoje)
data_atual = data_execucao.isoformat()
data_publico_ini = (data_execucao - timedelta(days=89)).isoformat()
data_publico_fim = (data_execucao + timedelta(days=1)).isoformat()

config_jdbc_db2 = {
    "fetchsize": 10_000,
}

if not isinstance(PERIODOS, int) or not 1 <= PERIODOS <= 6:
    raise ValueError("PERIODOS deve ser um número inteiro entre 1 e 6.")

print(f"Períodos financeiros fechados: {PERIODOS}")
print(f"Data de execução: {data_atual}")


## 3. Público-alvo

Seleciona clientes atualizados nas últimas 90 datas de calendário.


In [ ]:
%%spark

query_publico_contas = f"""
SELECT
    t.CD_CLI,
    t.NR_MCA_PCT_OPB,
    t.CD_PRD,
    t.NR_AG_TITR,
    t.CD_CT_TITR,
    MAX(t.TS_ATL_TRAN) AS TS_ATL_TRAN_CONTA,
    a.DD_INC_MM_CLC_BLC
FROM DB2GFP.TRAN_RLZD_INST_PCT t
LEFT JOIN DB2GFP.CT_GRDR_FNCO a
    ON t.NR_MCA_PCT_OPB = 999999999
   AND t.CD_PRD = 6
   AND t.NR_AG_TITR IS NOT NULL
   AND t.CD_CT_TITR IS NOT NULL
   AND a.CD_UOR_CC = t.NR_AG_TITR
   AND a.NR_CC = CASE
       WHEN t.NR_MCA_PCT_OPB = 999999999
        AND t.CD_PRD = 6
        AND t.CD_CT_TITR IS NOT NULL
        AND TRIM(t.CD_CT_TITR) <> ''
       THEN DECIMAL(TRIM(t.CD_CT_TITR), 31, 0)
   END
WHERE t.TS_ATL_TRAN >= DATE('{data_publico_ini}')
  AND t.TS_ATL_TRAN <  DATE('{data_publico_fim}')
  AND t.CD_EST_TRAN_INST = 0
GROUP BY
    t.CD_CLI,
    t.NR_MCA_PCT_OPB,
    t.CD_PRD,
    t.NR_AG_TITR,
    t.CD_CT_TITR,
    a.DD_INC_MM_CLC_BLC
"""

inicio_extracao = perf_counter()

df_publico_contas = cliente_db2.run_select(
    query_publico_contas,
    **config_jdbc_db2,
).persist()

qt_publico_contas = df_publico_contas.count()
duracao_extracao = perf_counter() - inicio_extracao

print(
    f"Extração DB2 público e contas: {qt_publico_contas:,} linhas "
    f"em {duracao_extracao:.2f}s"
)

df_publico_contas.createOrReplaceTempView(
    "vw_publico_contas"
)

query_publico_alvo = """
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    CAST(MAX(TS_ATL_TRAN_CONTA) AS TIMESTAMP) AS TS_ATL_TRAN
FROM vw_publico_contas
GROUP BY
    CAST(CD_CLI AS INT)
"""

df_publico_alvo = spark.sql(query_publico_alvo).persist()
qt_publico_alvo = df_publico_alvo.count()

if qt_publico_alvo == 0:
    raise ValueError("O recorte operacional de 90 dias não encontrou clientes.")

df_publico_alvo.createOrReplaceTempView(
    "vw_publico_alvo"
)


## 4. Dia do ciclo

Obtém o dia do balanço e preserva os diagnósticos `996`, `997` e `999`.


In [ ]:
%%spark

query_dia_ciclo = """
WITH contas_elegiveis AS (
    SELECT DISTINCT
        CAST(CD_CLI AS INT) AS CD_CLI,
        CAST(NR_AG_TITR AS INT) AS CD_UOR_CC,
        CAST(TRIM(CD_CT_TITR) AS DECIMAL(31,0)) AS NR_CC,
        CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC
    FROM vw_publico_contas
    WHERE NR_MCA_PCT_OPB = 999999999
      AND CD_PRD = 6
      AND NR_AG_TITR IS NOT NULL
      AND NULLIF(TRIM(CD_CT_TITR), '') IS NOT NULL
),
contas_cliente AS (
    SELECT
        p.CD_CLI,
        COUNT(c.CD_CLI) AS QT_CONTA_ELEGIVEL,
        MAX(c.DD_INC_MM_CLC_BLC) AS DD_INC_MM_CLC_BLC
    FROM vw_publico_alvo p
    LEFT JOIN contas_elegiveis c
        ON c.CD_CLI = p.CD_CLI
    GROUP BY
        p.CD_CLI
)
SELECT
    CD_CLI,
    CAST(
        CASE
            WHEN QT_CONTA_ELEGIVEL = 0 THEN 997
            WHEN QT_CONTA_ELEGIVEL > 1 THEN 996
            WHEN DD_INC_MM_CLC_BLC IS NULL THEN 999
            ELSE DD_INC_MM_CLC_BLC
        END
        AS SMALLINT
    ) AS DD_INC_MM_CLC_BLC
FROM contas_cliente
"""

df_dia_ciclo = spark.sql(query_dia_ciclo)
df_dia_ciclo.createOrReplaceTempView(
    "vw_dia_ciclo"
)


## 5. Janela financeira

Calcula os ciclos fechados individuais a partir de `TS_ATL_TRAN`.


In [ ]:
%%spark

query_janela_financeira = f"""
WITH parametros AS (
    SELECT
        CAST(p.CD_CLI AS INT) AS CD_CLI,
        CAST(p.TS_ATL_TRAN AS TIMESTAMP) AS TS_ATL_TRAN,
        CAST(d.DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC,
        CASE
            WHEN d.DD_INC_MM_CLC_BLC BETWEEN 1 AND 31
            THEN d.DD_INC_MM_CLC_BLC
            ELSE 1
        END AS DD_INC_MM_CLC_BLC_CALCULO,
        TRUNC(CAST(p.TS_ATL_TRAN AS DATE), 'MM') AS DT_MES_REFERENCIA
    FROM vw_publico_alvo p
    INNER JOIN vw_dia_ciclo d
        ON CAST(d.CD_CLI AS INT) = CAST(p.CD_CLI AS INT)
),
inicio_mes_referencia AS (
    SELECT
        p.*,
        DATE_ADD(
            DT_MES_REFERENCIA,
            LEAST(
                DD_INC_MM_CLC_BLC_CALCULO,
                DAY(LAST_DAY(DT_MES_REFERENCIA))
            ) - 1
        ) AS DT_INI_CICLO_MES_REFERENCIA
    FROM parametros p
),
inicio_ciclo_aberto AS (
    SELECT
        i.*,
        CASE
            WHEN CAST(TS_ATL_TRAN AS DATE) >= DT_INI_CICLO_MES_REFERENCIA
            THEN DT_INI_CICLO_MES_REFERENCIA
            ELSE DATE_ADD(
                ADD_MONTHS(DT_MES_REFERENCIA, -1),
                LEAST(
                    DD_INC_MM_CLC_BLC_CALCULO,
                    DAY(LAST_DAY(ADD_MONTHS(DT_MES_REFERENCIA, -1)))
                ) - 1
            )
        END AS DT_INI_CICLO_ABERTO
    FROM inicio_mes_referencia i
),
mes_inicio_janela AS (
    SELECT
        a.*,
        ADD_MONTHS(
            TRUNC(DT_INI_CICLO_ABERTO, 'MM'),
            -{PERIODOS}
        ) AS DT_MES_INI_JANELA
    FROM inicio_ciclo_aberto a
)
SELECT
    CD_CLI,
    TS_ATL_TRAN,
    DD_INC_MM_CLC_BLC,
    DD_INC_MM_CLC_BLC_CALCULO,
    CAST(
        DATE_ADD(
            DT_MES_INI_JANELA,
            LEAST(
                DD_INC_MM_CLC_BLC_CALCULO,
                DAY(LAST_DAY(DT_MES_INI_JANELA))
            ) - 1
        ) AS DATE
    ) AS DT_REF_INI,
    CAST(DATE_SUB(DT_INI_CICLO_ABERTO, 1) AS DATE) AS DT_REF_FIM
FROM mes_inicio_janela
"""

df_janela_financeira = spark.sql(query_janela_financeira)
df_janela_financeira.createOrReplaceTempView(
    "vw_janela_financeira"
)


## 6. Limites globais

Reduz a leitura DB2 ao menor e ao maior limite do público.


In [ ]:
%%spark

from pyspark.sql import functions as F

limites_janela = (
    df_janela_financeira
    .agg(
        F.min("DT_REF_INI").alias("DT_REF_INI_GLOBAL"),
        F.max("DT_REF_FIM").alias("DT_REF_FIM_GLOBAL"),
    )
    .first()
)

dt_ref_ini_global = limites_janela["DT_REF_INI_GLOBAL"].isoformat()
dt_ref_fim_global = limites_janela["DT_REF_FIM_GLOBAL"].isoformat()

print(f"Limite global inicial: {dt_ref_ini_global}")
print(f"Limite global final: {dt_ref_fim_global}")


## 7. Transações

Busca transações e aplica a janela individual de cada cliente.


In [ ]:
%%spark

query_transacoes_janela = f"""
SELECT
    t.NR_TRAN_INST_PCT,
    t.CD_CLI,
    t.DT_TRAN,
    t.VL_TRAN,
    t.CD_NTZ_CTB_TRAN,
    t.CD_CTGR_TRAN_OGNL
FROM DB2GFP.TRAN_RLZD_INST_PCT t
WHERE t.DT_TRAN >= DATE('{dt_ref_ini_global}')
  AND t.DT_TRAN <= DATE('{dt_ref_fim_global}')
  AND t.CD_EST_TRAN_INST = 0
  AND EXISTS (
      SELECT
          1
      FROM DB2GFP.TRAN_RLZD_INST_PCT p
      WHERE p.CD_CLI = t.CD_CLI
        AND p.TS_ATL_TRAN >= DATE('{data_publico_ini}')
        AND p.TS_ATL_TRAN <  DATE('{data_publico_fim}')
        AND p.CD_EST_TRAN_INST = 0
  )
"""

inicio_extracao = perf_counter()

df_transacoes_globais = cliente_db2.run_select(
    query_transacoes_janela,
    **config_jdbc_db2,
).persist()

qt_transacoes_janela = df_transacoes_globais.count()
duracao_extracao = perf_counter() - inicio_extracao

print(
    f"Extração DB2 transações da janela: {qt_transacoes_janela:,} linhas "
    f"em {duracao_extracao:.2f}s"
)

df_transacoes_globais.createOrReplaceTempView(
    "vw_transacoes_globais"
)

query_transacoes_cliente = """
SELECT
    t.*
FROM vw_transacoes_globais t
INNER JOIN vw_janela_financeira j
    ON CAST(j.CD_CLI AS INT) = CAST(t.CD_CLI AS INT)
   AND t.DT_TRAN BETWEEN j.DT_REF_INI AND j.DT_REF_FIM
"""

df_base_transacoes = spark.sql(query_transacoes_cliente)


## 8. Classificação

Aplica o dicionário incorporado e os fallbacks por natureza.


In [ ]:
%%spark

from pyspark.sql import functions as F

CATEGORIAS = {
    0: {
        'TIPO': None,
 
        'CD_GRUPO': 0,
        'TX_GRUPO': 'Sem categoria',
 
        'CD_CATEGORIA': 0,
        'TX_CATEGORIA': 'Sem categoria',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 0,
        'TX_CLASS_RADAR': 'Outras Entradas',
        'IN_AGRO': 'N',
        },
 
    83: {
        'TIPO': None,
 
        'CD_GRUPO': 0,
        'TX_GRUPO': 'Sem categoria',
 
        'CD_CATEGORIA': 83,
        'TX_CATEGORIA': 'Sem Categoria',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 0,
        'TX_CLASS_RADAR': 'Outras Entradas',
        'IN_AGRO': 'N',
        },
 
    1: {
        'TIPO': 'C',
 
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
 
        'CD_CATEGORIA': 1,
        'TX_CATEGORIA': 'Salário',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        },
 
    2: {
        'TIPO': 'C',
 
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
 
        'CD_CATEGORIA': 2,
        'TX_CATEGORIA': 'Vale Alimentação',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        },
 
    3: {
        'TIPO': 'C',
 
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
 
        'CD_CATEGORIA': 3,
        'TX_CATEGORIA': 'Restituição de IR',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 2,
        'TX_CLASS_RADAR': 'Estorno',
        'IN_AGRO': 'N',
        },
 
    4: {
        'TIPO': 'C',
 
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
 
        'CD_CATEGORIA': 4,
        'TX_CATEGORIA': 'Bonificação',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        },
 
    5: {
        'TIPO': 'C',
 
        'CD_GRUPO': 1,
        'TX_GRUPO': 'Receitas',
 
        'CD_CATEGORIA': 5,
        'TX_CATEGORIA': 'Outros Rendimentos',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'N',
        },
 
    6: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 6,
        'TX_CATEGORIA': 'Água',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    7: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 7,
        'TX_CATEGORIA': 'Eletricidade e Gás',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    9: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 9,
        'TX_CATEGORIA': 'Compra de Imóvel',
 
        'CD_IR': 2,
        'TX_IR': 'Bens e direitos',
 
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        },
 
    10: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 10,
        'TX_CATEGORIA': 'Aluguel e Condomínio',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    11: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 11,
        'TX_CATEGORIA': 'Móveis e Utensílios',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    12: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 12,
        'TX_CATEGORIA': 'Serviços e Manutenção',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    13: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 13,
        'TX_CATEGORIA': 'Empregados',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    14: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 14,
        'TX_CATEGORIA': 'Animais e Pets',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    3790: {
        'TIPO': 'D',
 
        'CD_GRUPO': 2,
        'TX_GRUPO': 'Casa',
 
        'CD_CATEGORIA': 3790,
        'TX_CATEGORIA': 'Seguro Residencial',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    15: {
        'TIPO': 'D',
 
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
 
        'CD_CATEGORIA': 15,
        'TX_CATEGORIA': 'Educação Superior',
 
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    16: {
        'TIPO': 'D',
 
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
 
        'CD_CATEGORIA': 16,
        'TX_CATEGORIA': 'Colégio',
 
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    17: {
        'TIPO': 'D',
 
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
 
        'CD_CATEGORIA': 17,
        'TX_CATEGORIA': 'Idiomas',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    18: {
        'TIPO': 'D',
 
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
 
        'CD_CATEGORIA': 18,
        'TX_CATEGORIA': 'Publicações e Papelaria',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    20: {
        'TIPO': 'D',
 
        'CD_GRUPO': 3,
        'TX_GRUPO': 'Educação',
 
        'CD_CATEGORIA': 20,
        'TX_CATEGORIA': 'Outros Gastos, Educação',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    21: {
        'TIPO': 'D',
 
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
 
        'CD_CATEGORIA': 21,
        'TX_CATEGORIA': 'Viagens e Lazer',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    22: {
        'TIPO': 'D',
 
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
 
        'CD_CATEGORIA': 22,
        'TX_CATEGORIA': 'Esportes e Academia',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    25: {
        'TIPO': 'D',
 
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
 
        'CD_CATEGORIA': 25,
        'TX_CATEGORIA': 'Cultura e Entretenimento',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    61: {
        'TIPO': 'D',
 
        'CD_GRUPO': 4,
        'TX_GRUPO': 'Lazer',
 
        'CD_CATEGORIA': 61,
        'TX_CATEGORIA': 'Jogos e Loterias',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    27: {
        'TIPO': 'D',
 
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
 
        'CD_CATEGORIA': 27,
        'TX_CATEGORIA': 'Plano de Saúde',
 
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    28: {
        'TIPO': 'D',
 
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
 
        'CD_CATEGORIA': 28,
        'TX_CATEGORIA': 'Serviços de Saúde',
 
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    29: {
        'TIPO': 'D',
 
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
 
        'CD_CATEGORIA': 29,
        'TX_CATEGORIA': 'Dentista',
 
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    30: {
        'TIPO': 'D',
 
        'CD_GRUPO': 5,
        'TX_GRUPO': 'Saúde',
 
        'CD_CATEGORIA': 30,
        'TX_CATEGORIA': 'Farmácias e Drogarias',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    32: {
        'TIPO': 'D',
 
        'CD_GRUPO': 6,
        'TX_GRUPO': 'Alimentação',
 
        'CD_CATEGORIA': 32,
        'TX_CATEGORIA': 'Feira e Supermercado',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    35: {
        'TIPO': 'D',
 
        'CD_GRUPO': 6,
        'TX_GRUPO': 'Alimentação',
 
        'CD_CATEGORIA': 35,
        'TX_CATEGORIA': 'Bar, Rest. e Padaria',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    36: {
        'TIPO': 'D',
 
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
 
        'CD_CATEGORIA': 36,
        'TX_CATEGORIA': 'Compra de Veículo',
 
        'CD_IR': 2,
        'TX_IR': 'Bens e direitos',
 
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        },
 
    37: {
        'TIPO': 'D',
 
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
 
        'CD_CATEGORIA': 37,
        'TX_CATEGORIA': 'Combustível',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    38: {
        'TIPO': 'D',
 
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
 
        'CD_CATEGORIA': 38,
        'TX_CATEGORIA': 'Estacionamento e Pedágio',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    39: {
        'TIPO': 'D',
 
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
 
        'CD_CATEGORIA': 39,
        'TX_CATEGORIA': 'Seguro de Veículo',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    40: {
        'TIPO': 'D',
 
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
 
        'CD_CATEGORIA': 40,
        'TX_CATEGORIA': 'Serviços e Manutenção',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    41: {
        'TIPO': 'D',
 
        'CD_GRUPO': 7,
        'TX_GRUPO': 'Transporte',
 
        'CD_CATEGORIA': 41,
        'TX_CATEGORIA': 'Transporte Urbano e Apps',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    42: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 42,
        'TX_CATEGORIA': 'Vestuário e Acessórios',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    43: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 43,
        'TX_CATEGORIA': 'Cuidado Pessoal e Beleza',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    44: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 44,
        'TX_CATEGORIA': 'Compras Diversas',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        },
 
    45: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 45,
        'TX_CATEGORIA': 'Pensão Alimentícia',
 
        'CD_IR': 1,
        'TX_IR': 'Pagamentos efetuados',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    46: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 46,
        'TX_CATEGORIA': 'Seguros e Previdência',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        },
 
    47: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 47,
        'TX_CATEGORIA': 'Doação',
 
        'CD_IR': 4,
        'TX_IR': 'Doações efetuadas',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    48: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 48,
        'TX_CATEGORIA': 'Gasto com Familiares',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    49: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 49,
        'TX_CATEGORIA': 'Presentes',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    60: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 60,
        'TX_CATEGORIA': 'Serviços Diversos',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    4417: {
        'TIPO': 'D',
 
        'CD_GRUPO': 8,
        'TX_GRUPO': 'Despesas Pessoais',
 
        'CD_CATEGORIA': 4417,
        'TX_CATEGORIA': 'Empréstimos e Prestações',
 
        'CD_IR': 3,
        'TX_IR': 'Dívidas e ônus reais',
 
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        },
 
    51: {
        'TIPO': 'D',
 
        'CD_GRUPO': 9,
        'TX_GRUPO': 'Comunicação',
 
        'CD_CATEGORIA': 51,
        'TX_CATEGORIA': 'Telefonia e Internet',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    53: {
        'TIPO': 'D',
 
        'CD_GRUPO': 9,
        'TX_GRUPO': 'Comunicação',
 
        'CD_CATEGORIA': 53,
        'TX_CATEGORIA': 'Assinatura TV e Streaming',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 7,
        'TX_CLASS_RADAR': 'Flexíveis',
        'IN_AGRO': 'N',
        },
 
    54: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 54,
        'TX_CATEGORIA': 'IPTU',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    55: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 55,
        'TX_CATEGORIA': 'IPVA e Gastos Detran',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    56: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 56,
        'TX_CATEGORIA': 'Imposto de Renda',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    57: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 57,
        'TX_CATEGORIA': 'ISS(Imposto sobre Serviços)',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 6,
        'TX_CLASS_RADAR': 'Essenciais',
        'IN_AGRO': 'N',
        },
 
    58: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 58,
        'TX_CATEGORIA': 'GPS(Guia de Previdência Social)',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 8,
        'TX_CLASS_RADAR': 'Futuro',
        'IN_AGRO': 'N',
        },
 
    59: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 59,
        'TX_CATEGORIA': 'Serviços Financeiros',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        },
 
    3787: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 3787,
        'TX_CATEGORIA': 'IOF',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        },
 
    3788: {
        'TIPO': 'D',
 
        'CD_GRUPO': 10,
        'TX_GRUPO': 'Tarifas e impostos',
 
        'CD_CATEGORIA': 3788,
        'TX_CATEGORIA': 'Encargos e Tarifas',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        },
 
    279: {
        'TIPO': 'D',
 
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
 
        'CD_CATEGORIA': 279,
        'TX_CATEGORIA': 'Gastos Diversos',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        },
 
    39434: {
        'TIPO': 'D',
 
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
 
        'CD_CATEGORIA': 39434,
        'TX_CATEGORIA': 'Cheque',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        },
 
    39435: {
        'TIPO': 'D',
 
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
 
        'CD_CATEGORIA': 39435,
        'TX_CATEGORIA': 'Saque',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        },
 
    39436: {
        'TIPO': 'D',
 
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
 
        'CD_CATEGORIA': 39436,
        'TX_CATEGORIA': 'Transferência',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        },
 
    39437: {
        'TIPO': 'D',
 
        'CD_GRUPO': 11,
        'TX_GRUPO': 'Outros',
 
        'CD_CATEGORIA': 39437,
        'TX_CATEGORIA': 'Boletos Diversos',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'N',
        },
 
    111: {
        'TIPO': 'D',
 
        'CD_GRUPO': 12,
        'TX_GRUPO': 'Fatura',
 
        'CD_CATEGORIA': 111,
        'TX_CATEGORIA': 'Cartão de Crédito',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 9,
        'TX_CLASS_RADAR': 'Obrigações',
        'IN_AGRO': 'N',
        },
 
    448977: {
        'TIPO': 'D',
 
        'CD_GRUPO': 13,
        'TX_GRUPO': 'Investimentos',
 
        'CD_CATEGORIA': 448977,
        'TX_CATEGORIA': 'Aplicação',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 8,
        'TX_CLASS_RADAR': 'Futuro',
        'IN_AGRO': 'N',
        },
 
    448978: {
        'TIPO': 'C',
 
        'CD_GRUPO': 13,
        'TX_GRUPO': 'Investimentos',
 
        'CD_CATEGORIA': 448978,
        'TX_CATEGORIA': 'Resgate de Investimentos',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 3,
        'TX_CLASS_RADAR': 'Resgate',
        'IN_AGRO': 'N',
        },
 
    300: {
        'TIPO': 'C',
 
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
 
        'CD_CATEGORIA': 300,
        'TX_CATEGORIA': 'Receitas Agro',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 1,
        'TX_CLASS_RADAR': 'Renda',
        'IN_AGRO': 'S',
        },
 
    310: {
        'TIPO': 'D',
 
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
 
        'CD_CATEGORIA': 310,
        'TX_CATEGORIA': 'Criações',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        },
 
    330: {
        'TIPO': 'D',
 
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
 
        'CD_CATEGORIA': 330,
        'TX_CATEGORIA': 'Cultivos',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        },
 
    350: {
        'TIPO': 'D',
 
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
 
        'CD_CATEGORIA': 350,
        'TX_CATEGORIA': 'Insumos',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        },
 
    370: {
        'TIPO': 'D',
 
        'CD_GRUPO': 14,
        'TX_GRUPO': 'Agro',
 
        'CD_CATEGORIA': 370,
        'TX_CATEGORIA': 'Apoio Produtivo',
 
        'CD_IR': 0,
        'TX_IR': 'Não pertence',
 
        'CD_CLASS_RADAR': 5,
        'TX_CLASS_RADAR': 'Indeterminado',
        'IN_AGRO': 'S',
        },
 
}


# ============================================================
# Conversão do dicionário incorporado
# ============================================================

df_mapa_classificacao_categoria = spark.createDataFrame(
    [
        (
            int(cd_categoria),
            categoria["TIPO"],
            int(categoria["CD_CLASS_RADAR"]),
            categoria["TX_CLASS_RADAR"],
            categoria["IN_AGRO"],
        )
        for cd_categoria, categoria in sorted(CATEGORIAS.items())
        if categoria["TIPO"] in ("C", "D")
    ],
    """
        CD_CATEGORIA BIGINT,
        TIPO STRING,
        CD_CLASS_RADAR BIGINT,
        TX_CLASS_RADAR STRING,
        IN_AGRO STRING
    """,
)

df_mapa_classificacao_categoria.createOrReplaceTempView(
    "vw_mapa_classificacao_categoria"
)


# ============================================================
# Aplicação da classificação na base transacional
# ============================================================

df_base_transacoes = (
    df_base_transacoes.alias("b")
    .join(
        F.broadcast(df_mapa_classificacao_categoria).alias("m"),
        (
            F.col("b.CD_CTGR_TRAN_OGNL").cast("bigint")
            == F.col("m.CD_CATEGORIA")
        )
        & (
            F.col("b.CD_NTZ_CTB_TRAN")
            == F.col("m.TIPO")
        ),
        "left",
    )
    .select(
        "b.*",
        F.coalesce(
            F.col("m.CD_CLASS_RADAR"),
            F.when(F.col("b.CD_NTZ_CTB_TRAN") == "C", F.lit(0))
            .when(F.col("b.CD_NTZ_CTB_TRAN") == "D", F.lit(5))
            .otherwise(F.lit(None).cast("bigint")),
        ).cast("bigint").alias("CD_CLASSIFICACAO_CATEGORIA"),
        F.coalesce(
            F.col("m.TX_CLASS_RADAR"),
            F.when(
                F.col("b.CD_NTZ_CTB_TRAN") == "C",
                F.lit("Outras Entradas"),
            )
            .when(
                F.col("b.CD_NTZ_CTB_TRAN") == "D",
                F.lit("Indeterminado"),
            )
            .otherwise(F.lit(None).cast("string")),
        ).alias("NM_CLASSIFICACAO_CATEGORIA"),
        F.coalesce(
            F.col("m.IN_AGRO"),
            F.lit("N"),
        ).alias("FL_AGRO_CATEGORIA"),
    )
)

df_base_transacoes.createOrReplaceTempView(
    "vw_base_transacoes"
)


## 9. Perfil financeiro

Seleciona e nomeia o perfil financeiro mais recente.


In [ ]:
%%spark

query_perfil_financeiro = """
WITH perfil_mais_recente AS (
    SELECT
        CAST(CD_CLI AS BIGINT) AS CD_CLI,
        CAST(CD_MAC_PRFL_CLI AS BIGINT) AS CD_MAC_PRFL_CLI,
        CAST(CD_MIC_PRFL_CLI AS BIGINT) AS CD_MIC_PRFL_CLI,
        ROW_NUMBER() OVER (
            PARTITION BY CD_CLI
            ORDER BY
                CAST(DT_REF AS DATE) DESC NULLS LAST,
                CD_MAC_PRFL_CLI DESC,
                CD_MIC_PRFL_CLI DESC
        ) AS NR_ORDEM
    FROM sbx_t2i2016.DVS_GRDR_FNCO_PF
),
perfil_nomeado AS (
    SELECT
        CD_CLI,
        CD_MAC_PRFL_CLI,
        CASE
            WHEN CD_MAC_PRFL_CLI IS NULL THEN NULL
            WHEN CD_MAC_PRFL_CLI = 1 THEN 'Endividado'
            WHEN CD_MAC_PRFL_CLI = 2 THEN 'Equilibrista'
            WHEN CD_MAC_PRFL_CLI = 3 THEN 'Investidor'
            ELSE 'CODIGO NAO MAPEADO'
        END AS NM_MAC_PRFL_CLI,
        CD_MIC_PRFL_CLI,
        CASE
            WHEN CD_MIC_PRFL_CLI IS NULL THEN NULL
            WHEN CD_MIC_PRFL_CLI = 1 THEN 'Inadimplente'
            WHEN CD_MIC_PRFL_CLI = 2 THEN 'Acrobata'
            WHEN CD_MIC_PRFL_CLI = 3 THEN 'Iminente'
            WHEN CD_MIC_PRFL_CLI = 4 THEN 'Consciente'
            WHEN CD_MIC_PRFL_CLI = 5 THEN 'Equilibrista'
            WHEN CD_MIC_PRFL_CLI = 6 THEN 'Acelerado'
            WHEN CD_MIC_PRFL_CLI = 7 THEN 'Precavido'
            WHEN CD_MIC_PRFL_CLI = 8 THEN 'Despreocupado'
            ELSE 'CODIGO NAO MAPEADO'
        END AS NM_MIC_PRFL_CLI
    FROM perfil_mais_recente
    WHERE NR_ORDEM = 1
)
SELECT
    CD_CLI,
    CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI,
    CASE
        WHEN NM_MAC_PRFL_CLI IS NULL
          OR NM_MIC_PRFL_CLI IS NULL
        THEN CAST(NULL AS STRING)
        WHEN NM_MAC_PRFL_CLI = 'CODIGO NAO MAPEADO'
          OR NM_MIC_PRFL_CLI = 'CODIGO NAO MAPEADO'
        THEN 'CODIGO NAO MAPEADO'
        WHEN NM_MAC_PRFL_CLI = 'Equilibrista'
         AND NM_MIC_PRFL_CLI = 'Equilibrista'
        THEN 'Equilibrista'
        ELSE CONCAT(NM_MAC_PRFL_CLI, ' ', NM_MIC_PRFL_CLI)
    END AS NM_PRFL_FIN
FROM perfil_nomeado
"""

df_perfil_financeiro = spark.sql(query_perfil_financeiro)
df_perfil_financeiro.createOrReplaceTempView(
    "vw_perfil_financeiro"
)


## 10. Renda presumida

Mantém o contrato do dado pessoal até a definição da fonte.


In [ ]:
%%spark

# Substituir esta consulta quando a origem e a regra da renda presumida forem definidas.
query_renda_presumida = """
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    CAST(NULL AS DECIMAL(18,2)) AS VL_REN_PRES
FROM vw_publico_alvo
"""

df_renda_presumida = spark.sql(query_renda_presumida)
df_renda_presumida.createOrReplaceTempView(
    "vw_renda_presumida"
)


## 11. Agregações

Consolida quantidades, valores classificados e contexto agro.


In [ ]:
%%spark

query_agregacoes = """
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    COUNT(*) AS QT_TRANS_TOTAL,
    COUNT(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN 1 END) AS QT_TRANS_ENT,
    COUNT(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN 1 END) AS QT_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'C' THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_ENT,
    CAST(SUM(CASE WHEN CD_NTZ_CTB_TRAN = 'D' THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(25,2)) AS VL_TRANS_SAI,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 1 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_ENT_REN,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 2 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_ENT_EST,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 3 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_ENT_RESG,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 0 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_ENT_OUT,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 4 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_ENT_CRED,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 5 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_SAI_IND,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 6 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_SAI_ESS,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 7 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_SAI_FLEX,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 8 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_SAI_FUT,
    CAST(SUM(CASE WHEN CD_CLASSIFICACAO_CATEGORIA = 9 THEN COALESCE(VL_TRAN, 0) ELSE 0 END) AS DECIMAL(18,2)) AS VL_SAI_OBR,
    CASE
        WHEN MAX(CASE WHEN FL_AGRO_CATEGORIA = 'S' AND CD_NTZ_CTB_TRAN IN ('C', 'D') THEN 1 ELSE 0 END) = 1
        THEN 'S'
        ELSE 'N'
    END AS FL_TEM_MOV_AGRO
FROM vw_base_transacoes
GROUP BY
    CD_CLI
"""

df_agregacoes = spark.sql(query_agregacoes)
df_agregacoes.createOrReplaceTempView(
    "vw_agregacoes"
)


## 12. Base do cliente

Preserva todo o público, inclusive clientes sem movimento na janela.


In [ ]:
%%spark

query_base_cliente = """
SELECT
    j.CD_CLI,
    j.TS_ATL_TRAN,
    j.DD_INC_MM_CLC_BLC,
    r.VL_REN_PRES,
    j.DT_REF_INI,
    j.DT_REF_FIM,
    p.CD_MAC_PRFL_CLI,
    p.NM_MAC_PRFL_CLI,
    p.CD_MIC_PRFL_CLI,
    p.NM_MIC_PRFL_CLI,
    p.NM_PRFL_FIN,
    COALESCE(a.QT_TRANS_TOTAL, CAST(0 AS BIGINT)) AS QT_TRANS_TOTAL,
    COALESCE(a.QT_TRANS_ENT, CAST(0 AS BIGINT)) AS QT_TRANS_ENT,
    COALESCE(a.QT_TRANS_SAI, CAST(0 AS BIGINT)) AS QT_TRANS_SAI,
    COALESCE(a.VL_TRANS_ENT, CAST(0 AS DECIMAL(25,2))) AS VL_TRANS_ENT,
    COALESCE(a.VL_TRANS_SAI, CAST(0 AS DECIMAL(25,2))) AS VL_TRANS_SAI,
    COALESCE(a.VL_ENT_REN, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_REN,
    COALESCE(a.VL_ENT_EST, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_EST,
    COALESCE(a.VL_ENT_RESG, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_RESG,
    COALESCE(a.VL_ENT_OUT, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_OUT,
    COALESCE(a.VL_ENT_CRED, CAST(0 AS DECIMAL(18,2))) AS VL_ENT_CRED,
    COALESCE(a.VL_SAI_IND, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_IND,
    COALESCE(a.VL_SAI_ESS, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_ESS,
    COALESCE(a.VL_SAI_FLEX, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_FLEX,
    COALESCE(a.VL_SAI_FUT, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_FUT,
    COALESCE(a.VL_SAI_OBR, CAST(0 AS DECIMAL(18,2))) AS VL_SAI_OBR,
    COALESCE(a.FL_TEM_MOV_AGRO, 'N') AS FL_TEM_MOV_AGRO
FROM vw_janela_financeira j
LEFT JOIN vw_renda_presumida r
    ON r.CD_CLI = j.CD_CLI
LEFT JOIN vw_agregacoes a
    ON a.CD_CLI = j.CD_CLI
LEFT JOIN vw_perfil_financeiro p
    ON CAST(p.CD_CLI AS INT) = j.CD_CLI
"""

df_base_cliente = spark.sql(query_base_cliente)
df_base_cliente.createOrReplaceTempView(
    "vw_base_cliente"
)


## 13. Percentuais

Calcula totais, percentuais sobre entrada e parâmetros de referência.


In [ ]:
%%spark

query_percentuais = """
WITH totais AS (
    SELECT
        b.*,
        CAST(VL_ENT_REN + VL_ENT_EST + VL_ENT_RESG + VL_ENT_OUT + VL_ENT_CRED AS DECIMAL(18,2)) AS VL_ENT_TOTAL,
        CAST(VL_SAI_IND + VL_SAI_ESS + VL_SAI_FLEX + VL_SAI_FUT + VL_SAI_OBR AS DECIMAL(18,2)) AS VL_SAI_TOTAL
    FROM vw_base_cliente b
)
SELECT
    t.*,
    CAST(COALESCE(VL_SAI_TOTAL / NULLIF(VL_ENT_TOTAL, 0), 0) AS DECIMAL(9,6)) AS PC_SAI_ENT,
    CAST(COALESCE(VL_SAI_IND / NULLIF(VL_ENT_TOTAL, 0), 0) AS DECIMAL(9,6)) AS PC_SAI_IND,
    CAST(COALESCE(VL_SAI_ESS / NULLIF(VL_ENT_TOTAL, 0), 0) AS DECIMAL(9,6)) AS PC_SAI_ESS,
    CAST(COALESCE(VL_SAI_FLEX / NULLIF(VL_ENT_TOTAL, 0), 0) AS DECIMAL(9,6)) AS PC_SAI_FLEX,
    CAST(COALESCE(VL_SAI_FUT / NULLIF(VL_ENT_TOTAL, 0), 0) AS DECIMAL(9,6)) AS PC_SAI_FUT,
    CAST(COALESCE(VL_SAI_OBR / NULLIF(VL_ENT_TOTAL, 0), 0) AS DECIMAL(9,6)) AS PC_SAI_OBR,
    CAST(0.750000 AS DECIMAL(9,6)) AS PC_REF_IND,
    CAST(0.500000 AS DECIMAL(9,6)) AS PC_REF_ESS,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_FLEX,
    CAST(0.200000 AS DECIMAL(9,6)) AS PC_REF_FUT,
    CAST(0.300000 AS DECIMAL(9,6)) AS PC_REF_OBR
FROM totais t
"""

df_percentuais = spark.sql(query_percentuais)
df_percentuais.createOrReplaceTempView(
    "vw_percentuais"
)


## 14. Orçamento

Classifica o resultado orçamentário quando existe base transacional.


In [ ]:
%%spark

query_orcamento = """
WITH resultado AS (
    SELECT
        p.*,
        CAST(
            CASE
                WHEN QT_TRANS_TOTAL = 0 THEN NULL
                ELSE VL_ENT_TOTAL - VL_SAI_TOTAL
            END AS DECIMAL(18,2)
        ) AS VL_RES_ORC,
        CAST(
            CASE
                WHEN QT_TRANS_TOTAL = 0 THEN NULL
                WHEN PC_SAI_ENT BETWEEN 0.950000 AND 1.050000 THEN 0
                WHEN PC_SAI_ENT > 1.050000 AND PC_SAI_ENT <= 1.250000 THEN 1
                WHEN PC_SAI_ENT > 1.250000 THEN 2
                WHEN PC_SAI_ENT >= 0.750000 AND PC_SAI_ENT < 0.950000 THEN 3
                ELSE 4
            END AS INT
        ) AS CD_FAIXA_ORC
    FROM vw_percentuais p
)
SELECT
    r.*,
    CAST(
        CASE
            WHEN CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 0 THEN 0
            WHEN CD_FAIXA_ORC IN (3, 4) THEN 1
            ELSE 2
        END AS INT
    ) AS CD_RES_ORC,
    CASE
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC IN (3, 4) THEN 'Superavitário'
        ELSE 'Deficitário'
    END AS TX_RES_ORC,
    CASE
        WHEN CD_FAIXA_ORC IS NULL OR CD_FAIXA_ORC = 0 THEN NULL
        WHEN CD_FAIXA_ORC IN (2, 4) THEN 'Acentuado'
        ELSE 'Moderado'
    END AS TX_STS_RES,
    CASE
        WHEN CD_FAIXA_ORC IS NULL THEN NULL
        WHEN CD_FAIXA_ORC = 0 THEN 'Neutro'
        WHEN CD_FAIXA_ORC = 1 THEN 'Deficitário Moderado'
        WHEN CD_FAIXA_ORC = 2 THEN 'Deficitário Acentuado'
        WHEN CD_FAIXA_ORC = 3 THEN 'Superavitário Moderado'
        ELSE 'Superavitário Acentuado'
    END AS TX_STS_FINAL
FROM resultado r
"""

df_orcamento = spark.sql(query_orcamento)
df_orcamento.createOrReplaceTempView(
    "vw_orcamento"
)


## 15. Pontuação por concentração

Aplica as referências aos percentuais classificados.


In [ ]:
%%spark

query_pontuacao_concentracao = """
SELECT
    o.*,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_ENT_TOTAL = 0 THEN 0
            WHEN PC_SAI_IND > PC_REF_IND THEN 99
            ELSE 0
        END AS INT
    ) AS NR_PONT_CONC_IND,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_ENT_TOTAL = 0 THEN 0
            WHEN PC_SAI_ESS < PC_REF_ESS THEN 0
            WHEN PC_SAI_ESS < PC_REF_ESS * 1.5 THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_ENT_TOTAL = 0 THEN 0
            WHEN PC_SAI_FLEX < PC_REF_FLEX THEN 0
            WHEN PC_SAI_FLEX < PC_REF_FLEX * 1.5 THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_FLEX,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_ENT_TOTAL = 0 THEN 0
            WHEN PC_SAI_FUT >= PC_REF_FUT * 1.5 THEN 0
            WHEN PC_SAI_FUT >= PC_REF_FUT THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_FUT,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 THEN NULL
            WHEN VL_ENT_TOTAL = 0 THEN 0
            WHEN PC_SAI_OBR < PC_REF_OBR THEN 0
            WHEN PC_SAI_OBR < PC_REF_OBR * 1.5 THEN 1
            ELSE 2
        END AS INT
    ) AS NR_PONT_CONC_OBR
FROM vw_orcamento o
"""

df_pontuacao_concentracao = spark.sql(query_pontuacao_concentracao)
df_pontuacao_concentracao.createOrReplaceTempView(
    "vw_pontuacao_concentracao"
)


## 16. Pontuação orçamentária

Converte a faixa do orçamento em pontuação.


In [ ]:
%%spark

query_pontuacao_orcamento = """
SELECT
    c.*,
    CAST(
        CASE WHEN QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END
        AS INT
    ) AS NR_PONT_ORC_IND,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_FLEX,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 4 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 3) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_FUT,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0 OR CD_FAIXA_ORC IS NULL THEN NULL
            WHEN CD_FAIXA_ORC = 2 THEN 2
            WHEN CD_FAIXA_ORC IN (0, 1) THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_ORC_OBR
FROM vw_pontuacao_concentracao c
"""

df_pontuacao_orcamento = spark.sql(query_pontuacao_orcamento)
df_pontuacao_orcamento.createOrReplaceTempView(
    "vw_pontuacao_orcamento"
)


## 17. Pontuação por perfil

Calcula os pontos associados ao perfil financeiro.


In [ ]:
%%spark

query_pontuacao_perfil = """
SELECT
    o.*,
    CAST(
        CASE WHEN QT_TRANS_TOTAL = 0 THEN NULL ELSE 0 END
        AS INT
    ) AS NR_PONT_PRFL_IND,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0
              OR NM_PRFL_FIN IS NULL
              OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO'
            THEN NULL
            WHEN NM_PRFL_FIN = 'Endividado Acrobata' THEN 2
            WHEN NM_PRFL_FIN = 'Endividado Inadimplente'
              OR NM_PRFL_FIN IN (
                  'Equilibrista',
                  'Investidor Precavido',
                  'Investidor Despreocupado',
                  'Investidor Acelerado'
              )
            THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_ESS,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0
              OR NM_PRFL_FIN IS NULL
              OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO'
            THEN NULL
            WHEN NM_PRFL_FIN = 'Endividado Consciente' THEN 2
            WHEN NM_PRFL_FIN = 'Endividado Iminente' THEN 1
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_FLEX,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0
              OR NM_PRFL_FIN IS NULL
              OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO'
            THEN NULL
            WHEN NM_PRFL_FIN = 'Endividado Consciente' THEN 1
            WHEN NM_PRFL_FIN IN (
                'Equilibrista',
                'Investidor Precavido',
                'Investidor Despreocupado',
                'Investidor Acelerado'
            )
            THEN 2
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_FUT,
    CAST(
        CASE
            WHEN QT_TRANS_TOTAL = 0
              OR NM_PRFL_FIN IS NULL
              OR NM_PRFL_FIN = 'CODIGO NAO MAPEADO'
            THEN NULL
            WHEN NM_PRFL_FIN = 'Endividado Acrobata' THEN 1
            WHEN NM_PRFL_FIN IN (
                'Endividado Iminente',
                'Endividado Inadimplente'
            )
            THEN 2
            ELSE 0
        END AS INT
    ) AS NR_PONT_PRFL_OBR
FROM vw_pontuacao_orcamento o
"""

df_pontuacao_perfil = spark.sql(query_pontuacao_perfil)
df_pontuacao_perfil.createOrReplaceTempView(
    "vw_pontuacao_perfil"
)


## 18. Pontuação final

Consolida concentração, orçamento e perfil.


In [ ]:
%%spark

query_pontuacao_final = """
SELECT
    p.*,
    CAST(NR_PONT_CONC_IND AS INT) AS NR_PONT_IND_FIM,
    CAST(
        CASE
            WHEN NR_PONT_CONC_ESS IS NULL
              OR NR_PONT_ORC_ESS IS NULL
              OR NR_PONT_PRFL_ESS IS NULL
            THEN NULL
            ELSE NR_PONT_CONC_ESS + NR_PONT_ORC_ESS + NR_PONT_PRFL_ESS
        END AS INT
    ) AS NR_PONT_ESS_FIM,
    CAST(
        CASE
            WHEN NR_PONT_CONC_FLEX IS NULL
              OR NR_PONT_ORC_FLEX IS NULL
              OR NR_PONT_PRFL_FLEX IS NULL
            THEN NULL
            ELSE NR_PONT_CONC_FLEX + NR_PONT_ORC_FLEX + NR_PONT_PRFL_FLEX
        END AS INT
    ) AS NR_PONT_FLEX_FIM,
    CAST(
        CASE
            WHEN NR_PONT_CONC_FUT IS NULL
              OR NR_PONT_ORC_FUT IS NULL
              OR NR_PONT_PRFL_FUT IS NULL
            THEN NULL
            ELSE NR_PONT_CONC_FUT + NR_PONT_ORC_FUT + NR_PONT_PRFL_FUT
        END AS INT
    ) AS NR_PONT_FUT_FIM,
    CAST(
        CASE
            WHEN NR_PONT_CONC_OBR IS NULL
              OR NR_PONT_ORC_OBR IS NULL
              OR NR_PONT_PRFL_OBR IS NULL
            THEN NULL
            ELSE NR_PONT_CONC_OBR + NR_PONT_ORC_OBR + NR_PONT_PRFL_OBR
        END AS INT
    ) AS NR_PONT_OBR_FIM
FROM vw_pontuacao_perfil p
"""

df_pontuacao_final = spark.sql(query_pontuacao_final)
df_pontuacao_final.createOrReplaceTempView(
    "vw_pontuacao_final"
)


## 19. Vencedor

Define o conceito vencedor somente com cinco pontuações completas.


In [ ]:
%%spark

query_vencedor = """
WITH pontuacoes_completas AS (
    SELECT
        p.*,
        CASE
            WHEN NR_PONT_IND_FIM IS NULL
              OR NR_PONT_ESS_FIM IS NULL
              OR NR_PONT_FLEX_FIM IS NULL
              OR NR_PONT_FUT_FIM IS NULL
              OR NR_PONT_OBR_FIM IS NULL
            THEN NULL
            ELSE GREATEST(
                NR_PONT_IND_FIM,
                NR_PONT_ESS_FIM,
                NR_PONT_FLEX_FIM,
                NR_PONT_FUT_FIM,
                NR_PONT_OBR_FIM
            )
        END AS NR_PONT_MAX
    FROM vw_pontuacao_final p
),
codigo_vencedor AS (
    SELECT
        p.*,
        CASE
            WHEN NR_PONT_MAX IS NULL THEN NULL
            WHEN CAST(NR_PONT_IND_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_ESS_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_FLEX_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_FUT_FIM = NR_PONT_MAX AS INT)
               + CAST(NR_PONT_OBR_FIM = NR_PONT_MAX AS INT) > 1 THEN 9
            WHEN NR_PONT_IND_FIM = NR_PONT_MAX THEN 1
            WHEN NR_PONT_ESS_FIM = NR_PONT_MAX THEN 2
            WHEN NR_PONT_FLEX_FIM = NR_PONT_MAX THEN 3
            WHEN NR_PONT_FUT_FIM = NR_PONT_MAX THEN 4
            ELSE 5
        END AS CD_TEMA_VENCEDOR
    FROM pontuacoes_completas p
)
SELECT
    c.*,
    CASE CD_TEMA_VENCEDOR
        WHEN 1 THEN 'Categorização dos Gastos'
        WHEN 2 THEN 'Gestão de Orçamento'
        WHEN 3 THEN 'Consumo Planejado'
        WHEN 4 THEN 'Formação de Reserva'
        WHEN 5 THEN 'Uso Consciente do Crédito'
        WHEN 9 THEN 'Empate'
    END AS TX_TEMA_VENCEDOR
FROM codigo_vencedor c
"""

df_vencedor = spark.sql(query_vencedor)
df_vencedor.createOrReplaceTempView(
    "vw_vencedor"
)


## 20. Tabela final

Ordena e tipa as 71 colunas do contrato oficial.


In [ ]:
%%spark

query_tabela_final = f"""
SELECT
    CAST(CD_CLI AS INT) AS CD_CLI,
    CAST(TS_ATL_TRAN AS TIMESTAMP) AS TS_ATL_TRAN,
    CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC,
    CAST(VL_REN_PRES AS DECIMAL(18,2)) AS VL_REN_PRES,
    CD_MAC_PRFL_CLI,
    NM_MAC_PRFL_CLI,
    CD_MIC_PRFL_CLI,
    NM_MIC_PRFL_CLI,
    NM_PRFL_FIN,
    DT_REF_INI,
    DT_REF_FIM,
    TRUNC(CAST('{data_atual}' AS DATE), 'MM') AS DT_MES_EXEA,
    CAST('{data_atual}' AS DATE) AS DT_EXEA,
    QT_TRANS_TOTAL,
    QT_TRANS_ENT,
    QT_TRANS_SAI,
    VL_TRANS_ENT,
    VL_TRANS_SAI,
    VL_ENT_REN,
    VL_ENT_EST,
    VL_ENT_RESG,
    VL_ENT_OUT,
    VL_ENT_CRED,
    VL_ENT_TOTAL,
    VL_SAI_IND,
    VL_SAI_ESS,
    VL_SAI_FLEX,
    VL_SAI_FUT,
    VL_SAI_OBR,
    VL_SAI_TOTAL,
    VL_RES_ORC,
    PC_SAI_ENT,
    CD_RES_ORC,
    TX_RES_ORC,
    CD_FAIXA_ORC,
    TX_STS_RES,
    TX_STS_FINAL,
    PC_SAI_IND,
    PC_SAI_ESS,
    PC_SAI_FLEX,
    PC_SAI_FUT,
    PC_SAI_OBR,
    PC_REF_IND,
    PC_REF_ESS,
    PC_REF_FLEX,
    PC_REF_FUT,
    PC_REF_OBR,
    NR_PONT_CONC_IND,
    NR_PONT_CONC_ESS,
    NR_PONT_CONC_FLEX,
    NR_PONT_CONC_FUT,
    NR_PONT_CONC_OBR,
    NR_PONT_ORC_IND,
    NR_PONT_ORC_ESS,
    NR_PONT_ORC_FLEX,
    NR_PONT_ORC_FUT,
    NR_PONT_ORC_OBR,
    NR_PONT_PRFL_IND,
    NR_PONT_PRFL_ESS,
    NR_PONT_PRFL_FLEX,
    NR_PONT_PRFL_FUT,
    NR_PONT_PRFL_OBR,
    NR_PONT_IND_FIM,
    NR_PONT_ESS_FIM,
    NR_PONT_FLEX_FIM,
    NR_PONT_FUT_FIM,
    NR_PONT_OBR_FIM,
    CAST(CD_TEMA_VENCEDOR AS INT) AS CD_TEMA_VENCEDOR,
    TX_TEMA_VENCEDOR,
    FL_TEM_MOV_AGRO,
    CAST(NULL AS STRING) AS FL_PARTICIPA_RADAR
FROM vw_vencedor
"""

df_ana_edu_fin_cli = spark.sql(query_tabela_final)


## 21. DDL

Cria a tabela Hive conforme o contrato oficial.


In [ ]:
%%spark

atualizar_metadado = True

ddl_tabela_spark = f"""
CREATE TABLE {tabela_spark} (

    -- Bloco 1: Dados do cliente
    CD_CLI                     INT             COMMENT 'Código do cliente',
    TS_ATL_TRAN                TIMESTAMP       COMMENT 'Maior timestamp de atualização das transações do cliente no recorte operacional de 90 dias',
    DD_INC_MM_CLC_BLC          SMALLINT        COMMENT 'Dia inicial do cálculo do balanço: 1 a 31; 996 = múltiplas contas elegíveis; 997 = sem conta BB corrente identificável; 999 = conta sem data cadastrada',
    VL_REN_PRES                DECIMAL(18,2)   COMMENT 'Valor da renda presumida do cliente; nulo até a definição da fonte e da regra de preenchimento',
    CD_MAC_PRFL_CLI            BIGINT          COMMENT 'Código do macroperfil financeiro',
    NM_MAC_PRFL_CLI            STRING          COMMENT 'Texto do macroperfil financeiro; CODIGO NAO MAPEADO quando o código estiver fora do domínio conhecido',
    CD_MIC_PRFL_CLI            BIGINT          COMMENT 'Código do microperfil financeiro',
    NM_MIC_PRFL_CLI            STRING          COMMENT 'Texto do microperfil financeiro; CODIGO NAO MAPEADO quando o código estiver fora do domínio conhecido',
    NM_PRFL_FIN                STRING          COMMENT 'Texto unificado do perfil financeiro; nulo sem código e CODIGO NAO MAPEADO para código fora do domínio conhecido',

    -- Bloco 2: Período da análise
    DT_REF_INI                 DATE            COMMENT 'Menor data inicial entre os ciclos financeiros fechados selecionados',
    DT_REF_FIM                 DATE            COMMENT 'Maior data final entre os ciclos financeiros fechados selecionados',
    DT_MES_EXEA                DATE            COMMENT 'Mês de execução do ETL, representado pelo primeiro dia do mês',
    DT_EXEA                    DATE            COMMENT 'Data de execução do ETL',

    -- Bloco 3: Resumo técnico das transações
    QT_TRANS_TOTAL             BIGINT          COMMENT 'Quantidade total de transações',
    QT_TRANS_ENT               BIGINT          COMMENT 'Quantidade de transações de entrada',
    QT_TRANS_SAI               BIGINT          COMMENT 'Quantidade de transações de saída',
    VL_TRANS_ENT               DECIMAL(25,2)   COMMENT 'Valor total de entradas',
    VL_TRANS_SAI               DECIMAL(25,2)   COMMENT 'Valor total de saídas',

    -- Bloco 4: Valores de entrada
    VL_ENT_REN                 DECIMAL(18,2)   COMMENT 'Valores recebidos que representam renda, remuneração ou benefícios',
    VL_ENT_EST                 DECIMAL(18,2)   COMMENT 'Valores devolvidos ou recebidos de volta por correções, cancelamentos ou ajustes',
    VL_ENT_RESG                DECIMAL(18,2)   COMMENT 'Valores recuperados de investimentos ou aplicações financeiras',
    VL_ENT_OUT                 DECIMAL(18,2)   COMMENT 'Valores recebidos cuja origem não foi identificada ou não se enquadram nas demais classificações',
    VL_ENT_CRED                DECIMAL(18,2)   COMMENT 'Valores obtidos por empréstimos, financiamentos ou outras operações de crédito',
    VL_ENT_TOTAL               DECIMAL(18,2)   COMMENT 'Valor total de entrada',

    -- Bloco 5: Valores de saída
    VL_SAI_IND                 DECIMAL(18,2)   COMMENT 'Saídas cujo destino não foi identificado ou não se enquadram nas demais classificações',
    VL_SAI_ESS                 DECIMAL(18,2)   COMMENT 'Gastos necessários para a manutenção da vida e do dia a dia',
    VL_SAI_FLEX                DECIMAL(18,2)   COMMENT 'Gastos relacionados a escolhas pessoais, lazer e estilo de vida',
    VL_SAI_FUT                 DECIMAL(18,2)   COMMENT 'Valores destinados à formação de patrimônio, reserva ou objetivos futuros',
    VL_SAI_OBR                 DECIMAL(18,2)   COMMENT 'Valores destinados ao pagamento de dívidas, parcelas e compromissos financeiros',
    VL_SAI_TOTAL               DECIMAL(18,2)   COMMENT 'Valor total de saída',

    -- Bloco 6: Indicadores de orçamento
    VL_RES_ORC                 DECIMAL(18,2)   COMMENT 'Valor do resultado do orçamento: entradas menos saídas',
    PC_SAI_ENT                 DECIMAL(9,6)    COMMENT 'Percentual do valor total de saídas sobre o valor total de entradas: VL_SAI_TOTAL / VL_ENT_TOTAL',
    CD_RES_ORC                 INT             COMMENT 'Código do resultado do orçamento: 0 = Neutro; 1 = Superavitário; 2 = Deficitário',
    TX_RES_ORC                 STRING          COMMENT 'Texto do resultado do orçamento',
    CD_FAIXA_ORC               INT             COMMENT 'Código da faixa do resultado orçamentário: 0 = Neutro; 1 = Deficitário Moderado; 2 = Deficitário Acentuado; 3 = Superavitário Moderado; 4 = Superavitário Acentuado',
    TX_STS_RES                 STRING          COMMENT 'Status da intensidade do resultado: Acentuado ou Moderado; nulo quando o resultado for Neutro',
    TX_STS_FINAL               STRING          COMMENT 'Texto final composto pelo resultado e seu status',

    -- Bloco 7: Indicadores de saída sobre entrada
    PC_SAI_IND                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída indeterminada sobre o valor total de entradas: VL_SAI_IND / VL_ENT_TOTAL',
    PC_SAI_ESS                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída essencial sobre o valor total de entradas: VL_SAI_ESS / VL_ENT_TOTAL',
    PC_SAI_FLEX                DECIMAL(9,6)    COMMENT 'Percentual do valor de saída flexível sobre o valor total de entradas: VL_SAI_FLEX / VL_ENT_TOTAL',
    PC_SAI_FUT                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída para o futuro sobre o valor total de entradas: VL_SAI_FUT / VL_ENT_TOTAL',
    PC_SAI_OBR                 DECIMAL(9,6)    COMMENT 'Percentual do valor de saída para obrigações sobre o valor total de entradas: VL_SAI_OBR / VL_ENT_TOTAL',

    -- Bloco 8: Parâmetros de referência
    PC_REF_IND                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída indeterminada',
    PC_REF_ESS                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída essencial',
    PC_REF_FLEX                DECIMAL(9,6)    COMMENT 'Percentual de referência para saída flexível',
    PC_REF_FUT                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída destinada ao futuro',
    PC_REF_OBR                 DECIMAL(9,6)    COMMENT 'Percentual de referência para saída de obrigações',

    -- Bloco 9: Pontuação por concentração
    NR_PONT_CONC_IND           INT             COMMENT 'Pontuação de concentração da saída indeterminada',
    NR_PONT_CONC_ESS           INT             COMMENT 'Pontuação de concentração essencial',
    NR_PONT_CONC_FLEX          INT             COMMENT 'Pontuação de concentração flexível',
    NR_PONT_CONC_FUT           INT             COMMENT 'Pontuação de concentração da saída destinada ao futuro',
    NR_PONT_CONC_OBR           INT             COMMENT 'Pontuação de concentração da saída de obrigações',

    -- Bloco 10: Pontuação orçamentária
    NR_PONT_ORC_IND            INT             COMMENT 'Pontuação orçamentária da classificação indeterminada',
    NR_PONT_ORC_ESS            INT             COMMENT 'Pontuação de orçamento essencial',
    NR_PONT_ORC_FLEX           INT             COMMENT 'Pontuação de orçamento flexível',
    NR_PONT_ORC_FUT            INT             COMMENT 'Pontuação orçamentária da classificação futuro',
    NR_PONT_ORC_OBR            INT             COMMENT 'Pontuação orçamentária da classificação obrigações',

    -- Bloco 11: Pontuação por perfil
    NR_PONT_PRFL_IND           INT             COMMENT 'Pontuação por perfil da classificação indeterminada',
    NR_PONT_PRFL_ESS           INT             COMMENT 'Pontuação de perfil essencial',
    NR_PONT_PRFL_FLEX          INT             COMMENT 'Pontuação de perfil flexível',
    NR_PONT_PRFL_FUT           INT             COMMENT 'Pontuação por perfil da classificação futuro',
    NR_PONT_PRFL_OBR           INT             COMMENT 'Pontuação por perfil da classificação obrigações',

    -- Bloco 12: Pontuação consolidada e classificação vencedora
    NR_PONT_IND_FIM            INT             COMMENT 'Pontuação final da classificação indeterminada',
    NR_PONT_ESS_FIM            INT             COMMENT 'Pontuação final da classificação essenciais',
    NR_PONT_FLEX_FIM           INT             COMMENT 'Pontuação final da classificação flexíveis',
    NR_PONT_FUT_FIM            INT             COMMENT 'Pontuação final da classificação futuro',
    NR_PONT_OBR_FIM            INT             COMMENT 'Pontuação final da classificação obrigações',
    CD_TEMA_VENCEDOR           INT             COMMENT 'Código do conceito vencedor: 1 = Categorização dos Gastos; 2 = Gestão de Orçamento; 3 = Consumo Planejado; 4 = Formação de Reserva; 5 = Uso Consciente do Crédito; 9 = Empate; nulo sem pontuação completa',
    TX_TEMA_VENCEDOR           STRING          COMMENT 'Texto do conceito vencedor; nulo quando as cinco pontuações finais não estiverem preenchidas',

    -- Bloco 13: Contexto e elegibilidade
    FL_TEM_MOV_AGRO            STRING          COMMENT 'Indica se o cliente teve movimentação de crédito ou débito em categoria marcada como agro: S ou N',
    FL_PARTICIPA_RADAR         STRING          COMMENT 'Campo reservado para regra futura de participação; nulo nesta versão'
)
COMMENT 'Análise de Educação Financeira do Cliente'
STORED AS PARQUET
TBLPROPERTIES (
    'parquet.compress' = 'SNAPPY'
)
"""

query_drop_tabela = f"DROP TABLE IF EXISTS {tabela_spark}"

if atualizar_metadado:
    spark.sql(query_drop_tabela)
    spark.sql(ddl_tabela_spark)


## 22. Carga

Publica a tabela completa com `overwrite`.


In [ ]:
%%spark

(
    df_ana_edu_fin_cli
    .write
    .mode("overwrite")
    .insertInto(tabela_spark)
)

print(f"Tabela carregada com sucesso: {tabela_spark}")


In [ ]:
# comentario. nao remover
# %%spark

# query_perfil_origem = f"""
# SELECT
#     p.CD_CLI,
#     p.CD_MAC_PRFL_CLI,
#     p.CD_MIC_PRFL_CLI,
#     p.DT_REF
# FROM SCHEMA_A_DEFINIR.DVS_GRDR_FNCO_PF p
# WHERE EXISTS (
#     SELECT
#         1
#     FROM DB2GFP.TRAN_RLZD_INST_PCT t
#     WHERE t.CD_CLI = p.CD_CLI
#       AND t.TS_ATL_TRAN >= DATE('{data_publico_ini}')
#       AND t.TS_ATL_TRAN <  DATE('{data_publico_fim}')
#       AND t.CD_EST_TRAN_INST = 0
# )
# """

# inicio_extracao = perf_counter()

# df_perfil_origem = cliente_db2.run_select(
#     query_perfil_origem,
#     **config_jdbc_db2,
# ).persist()

# qt_perfil_origem = df_perfil_origem.count()
# duracao_extracao = perf_counter() - inicio_extracao

# print(
#     f"perfil_origem: {qt_perfil_origem} linhas "
#     f"em {duracao_extracao:.2f}s"
# )

# df_perfil_origem.createOrReplaceTempView(
#     "vw_perfil_origem"
# )

# query_perfil_financeiro = """
# WITH perfil_mais_recente AS (
#     SELECT
#         CAST(CD_CLI AS BIGINT) AS CD_CLI,
#         CAST(CD_MAC_PRFL_CLI AS BIGINT) AS CD_MAC_PRFL_CLI,
#         CAST(CD_MIC_PRFL_CLI AS BIGINT) AS CD_MIC_PRFL_CLI,
#         ROW_NUMBER() OVER (
#             PARTITION BY CD_CLI
#             ORDER BY
#                 CAST(DT_REF AS DATE) DESC NULLS LAST,
#                 CD_MAC_PRFL_CLI DESC,
#                 CD_MIC_PRFL_CLI DESC
#         ) AS NR_ORDEM
#     FROM vw_perfil_origem
# ),
# perfil_nomeado AS (
#     SELECT
#         CD_CLI,
#         CD_MAC_PRFL_CLI,
#         CASE
#             WHEN CD_MAC_PRFL_CLI IS NULL THEN NULL
#             WHEN CD_MAC_PRFL_CLI = 1 THEN 'Endividado'
#             WHEN CD_MAC_PRFL_CLI = 2 THEN 'Equilibrista'
#             WHEN CD_MAC_PRFL_CLI = 3 THEN 'Investidor'
#             ELSE 'CODIGO NAO MAPEADO'
#         END AS NM_MAC_PRFL_CLI,
#         CD_MIC_PRFL_CLI,
#         CASE
#             WHEN CD_MIC_PRFL_CLI IS NULL THEN NULL
#             WHEN CD_MIC_PRFL_CLI = 1 THEN 'Inadimplente'
#             WHEN CD_MIC_PRFL_CLI = 2 THEN 'Acrobata'
#             WHEN CD_MIC_PRFL_CLI = 3 THEN 'Iminente'
#             WHEN CD_MIC_PRFL_CLI = 4 THEN 'Consciente'
#             WHEN CD_MIC_PRFL_CLI = 5 THEN 'Equilibrista'
#             WHEN CD_MIC_PRFL_CLI = 6 THEN 'Acelerado'
#             WHEN CD_MIC_PRFL_CLI = 7 THEN 'Precavido'
#             WHEN CD_MIC_PRFL_CLI = 8 THEN 'Despreocupado'
#             ELSE 'CODIGO NAO MAPEADO'
#         END AS NM_MIC_PRFL_CLI
#     FROM perfil_mais_recente
#     WHERE NR_ORDEM = 1
# )
# SELECT
#     CD_CLI,
#     CD_MAC_PRFL_CLI,
#     NM_MAC_PRFL_CLI,
#     CD_MIC_PRFL_CLI,
#     NM_MIC_PRFL_CLI,
#     CASE
#         WHEN NM_MAC_PRFL_CLI IS NULL
#           OR NM_MIC_PRFL_CLI IS NULL
#         THEN CAST(NULL AS STRING)
#         WHEN NM_MAC_PRFL_CLI = 'CODIGO NAO MAPEADO'
#           OR NM_MIC_PRFL_CLI = 'CODIGO NAO MAPEADO'
#         THEN 'CODIGO NAO MAPEADO'
#         WHEN NM_MAC_PRFL_CLI = 'Equilibrista'
#          AND NM_MIC_PRFL_CLI = 'Equilibrista'
#         THEN 'Equilibrista'
#         ELSE CONCAT(NM_MAC_PRFL_CLI, ' ', NM_MIC_PRFL_CLI)
#     END AS NM_PRFL_FIN
# FROM perfil_nomeado
# """

# df_perfil_financeiro = spark.sql(query_perfil_financeiro)
# df_perfil_financeiro.createOrReplaceTempView(
#     "vw_perfil_financeiro"
# )
